### Uniting POEM with BGC results
Here we take the processed result from POEM pipeline and merge it with the BGC results (BIG-SCAPE, antiSMASH and DeepSEA results).

### Part 1 - Pre-processing of POEM results
This section has the objective to transform the dataframe from POEM_operon_information.csv, specificaly the "coordinate" column to represent the name_ids from CDS present in BGCs.

In [20]:
import pandas as pd
import glob
from Bio import SeqIO
import os
import re

In [28]:
df_poem = pd.read_csv("/home/pedro/resultados_bgc/fna_dos_bgcs/POEM_operon_information.csv")
df_poem

,Unnamed: 0,metagenome_id,strand,coordinates
0,0,MGYG000296006_71,+,2227/3084 --> 3081/3860 --> 3873/4478 --> 4475...
1,1,MGYG000296008_2,+,3329/4378 --> 4378/5151 --> 5102/5692 --> 5734...
2,2,MGYG000296008_2,-,494/1084 <-- 1065/1799 <-- 1803/2372 <-- 2373/...
3,3,MGYG000296008_2,-,12714/13298 <-- 13295/13717 <-- 13756/14715 <-...
4,4,MGYG000296008_22,-,206/1723 <-- 1710/2171 <-- 2164/2628 <-- 2638/...
...,...,...,...,...
366,366,MGYG000296076_4,-,10481/11671 <-- 11671/12852
367,367,MGYG000296076_4,-,18379/19077 <-- 19088/19942
368,368,MGYG000296076_4,-,22959/23456 <-- 23503/24327 <-- 24362/24787
369,369,MGYG000296076_4,-,27735/28226 <-- 28236/28712


In [29]:
gbk_files = glob.glob("/home/pedro/backup/resultados/MGYG*/MGYG*_*.gbk", recursive=True)

dados = []

for gbk in gbk_files:
    for record in SeqIO.parse(gbk, "genbank"):
        for feature in record.features:
            if feature.type == "CDS":
                
                start = int(feature.location.start) + 1  
                end = int(feature.location.end)
                
                locus_tag = feature.qualifiers.get("locus_tag", ["NA"])[0]
                translation = feature.qualifiers.get("translation", ["NA"])[0]
                
                dados.append({
                    "arquivo": os.path.basename(gbk),
                    "contig": record.id,
                    "start": start,
                    "end": end,
                    "locus_tag": locus_tag,
                    "translation": translation
                })
df = pd.DataFrame(dados)
df = df.rename(columns={"contig": "metagenome_id"})
df

,arquivo,metagenome_id,start,end,locus_tag,translation
0,MGYG000296020_64.region001.gbk,MGYG000296020_64,1,639,ctg64_1,MPEEIYDIGIIGGGLAGLALAIESAKAGRKVILFEKETFPFHKVCG...
1,MGYG000296020_64.region001.gbk,MGYG000296020_64,643,1335,ctg64_2,MPNFNIRSREKELLDQPNIPKAELFQNLRELDTINRLLGGHQATLI...
2,MGYG000296020_64.region001.gbk,MGYG000296020_64,1328,2434,ctg64_3,MGNKTNTRIVSIGTAVPENVYLQTDLADWMHQRLDSNNTRMGRLFR...
3,MGYG000296020_64.region001.gbk,MGYG000296020_64,2475,3320,ctg64_4,MLDALKLMRIPFSIFLMPVFWFALIPLEDIPIAKSVFIFLIIHVLI...
4,MGYG000296020_64.region001.gbk,MGYG000296020_64,3302,4018,ctg64_5,MHDDILVMRYLLLSFYVLSLSFSQPLKAEEGTNAIVNSYALGSIYK...
...,...,...,...,...,...,...
1737,MGYG000296015_9.region001.gbk,MGYG000296015_9,19388,19951,ctg9_25,MRSASLFLGEFLRRPAEVASMVPSSPWLKRRIVAAASLADAHVVIE...
1738,MGYG000296015_9.region001.gbk,MGYG000296015_9,19954,20583,ctg9_26,MRALALISVIIGLPLLAACVTINVYFPAVAAEKAADRIIEDVWGPN...
1739,MGYG000296015_9.region001.gbk,MGYG000296015_9,20653,22680,ctg9_27,MAGAVIGITDAAANAELGIHIGQLSRNGWTVEDIQLDYSTSADTLA...
1740,MGYG000296015_9.region001.gbk,MGYG000296015_9,22845,23549,ctg9_28,MAPHEQAESTPSQAQARYSEPEIEAVYRAIAERRDMRHFVPAPVDP...


In [30]:
df["coordinates"] = df["start"].astype(str) + "/" + df["end"].astype(str)
df

,arquivo,metagenome_id,start,end,locus_tag,translation,coordinates
0,MGYG000296020_64.region001.gbk,MGYG000296020_64,1,639,ctg64_1,MPEEIYDIGIIGGGLAGLALAIESAKAGRKVILFEKETFPFHKVCG...,1/639
1,MGYG000296020_64.region001.gbk,MGYG000296020_64,643,1335,ctg64_2,MPNFNIRSREKELLDQPNIPKAELFQNLRELDTINRLLGGHQATLI...,643/1335
2,MGYG000296020_64.region001.gbk,MGYG000296020_64,1328,2434,ctg64_3,MGNKTNTRIVSIGTAVPENVYLQTDLADWMHQRLDSNNTRMGRLFR...,1328/2434
3,MGYG000296020_64.region001.gbk,MGYG000296020_64,2475,3320,ctg64_4,MLDALKLMRIPFSIFLMPVFWFALIPLEDIPIAKSVFIFLIIHVLI...,2475/3320
4,MGYG000296020_64.region001.gbk,MGYG000296020_64,3302,4018,ctg64_5,MHDDILVMRYLLLSFYVLSLSFSQPLKAEEGTNAIVNSYALGSIYK...,3302/4018
...,...,...,...,...,...,...,...
1737,MGYG000296015_9.region001.gbk,MGYG000296015_9,19388,19951,ctg9_25,MRSASLFLGEFLRRPAEVASMVPSSPWLKRRIVAAASLADAHVVIE...,19388/19951
1738,MGYG000296015_9.region001.gbk,MGYG000296015_9,19954,20583,ctg9_26,MRALALISVIIGLPLLAACVTINVYFPAVAAEKAADRIIEDVWGPN...,19954/20583
1739,MGYG000296015_9.region001.gbk,MGYG000296015_9,20653,22680,ctg9_27,MAGAVIGITDAAANAELGIHIGQLSRNGWTVEDIQLDYSTSADTLA...,20653/22680
1740,MGYG000296015_9.region001.gbk,MGYG000296015_9,22845,23549,ctg9_28,MAPHEQAESTPSQAQARYSEPEIEAVYRAIAERRDMRHFVPAPVDP...,22845/23549


In [42]:
teste = df[df["metagenome_id"] == "MGYG000296008_2"]
teste

,arquivo,metagenome_id,start,end,locus_tag,translation,coordinates
1487,MGYG000296008_2.region001.gbk,MGYG000296008_2,494,1084,ctg2_2,MDQKIFSFLLRAAASSNHHKLHEVIKTIGPIDIPISNLRFQQYLIK...,494/1084
1488,MGYG000296008_2.region001.gbk,MGYG000296008_2,1065,1799,ctg2_3,MPRLQKFLADAGIGSRRYCEELIKSGEVTVNGKTAEIGCTVEHTDE...,1065/1799
1489,MGYG000296008_2.region001.gbk,MGYG000296008_2,1803,2372,ctg2_4,MDNQNAIESILISSGRPVRLSEIKKLLNAQTINLELSEIRVVIREL...,1803/2372
1490,MGYG000296008_2.region001.gbk,MGYG000296008_2,2373,3167,ctg2_5,MKAAVQSELPMLKGERLASLPDDLFIPPDALELFLEDFSGPMDVLL...,2373/3167
1491,MGYG000296008_2.region001.gbk,MGYG000296008_2,3329,4378,ctg2_6,MIYNTDNTRIIEKFDLITPHEVIQKYPLNDDISELVFGTRNEVSQI...,3329/4378
1492,MGYG000296008_2.region001.gbk,MGYG000296008_2,4378,5151,ctg2_7,MTKLSVDLEIAQTLGISVALIIALAERENILNLGELKSIASRELSF...,4378/5151
1493,MGYG000296008_2.region001.gbk,MGYG000296008_2,5138,5692,ctg2_8,MTSNKEQFLDAINKVFLDLELAYHYQFYKVFSNEDKINSAKKLWAE...,5138/5692
1494,MGYG000296008_2.region001.gbk,MGYG000296008_2,5734,6615,ctg2_9,MSKNVRLSIYILIPLIVWMISGVFISEDVTKESKPKTLSSVVIEQS...,5734/6615
1495,MGYG000296008_2.region001.gbk,MGYG000296008_2,6612,9761,ctg2_10,MNQIVDFALAKGKTTLTVALLIILAGAFARSTIPVASDPSIQIPIV...,6612/9761
1496,MGYG000296008_2.region001.gbk,MGYG000296008_2,9780,9998,ctg2_11,MTKKNTTDFETSIKKLEAIVEKLEEGDLDLDTSLKSFEEGVALVKE...,9780/9998


In [43]:
import re


lookup_start = {
    (row.metagenome_id, str(row.start)): row.locus_tag
    for row in df.itertuples()
}

lookup_end = {
    (row.metagenome_id, str(row.end)): row.locus_tag
    for row in df.itertuples()
}

In [44]:
def substituir(row):
    texto = row["coordinates"]
    meta = row["metagenome_id"]
    
    def troca(match):
        coord = match.group(0)          # ex: "10481/11671"
        start, end = coord.split("/")
        
        # 1️⃣ tenta por start
        gene = lookup_start.get((meta, start))
        if gene:
            return gene
        
        # 2️⃣ tenta por end
        gene = lookup_end.get((meta, end))
        if gene:
            return gene
        
        # 3️⃣ mantém original
        return coord
    
    return re.sub(r"\d+/\d+", troca, texto)

df_poem["coordinates"] = df_poem.apply(substituir, axis=1)

In [45]:
df_poem

,Unnamed: 0,metagenome_id,strand,coordinates
0,0,MGYG000296006_71,+,ctg71_3 --> ctg71_4 --> ctg71_5 --> ctg71_6 --...
1,1,MGYG000296008_2,+,ctg2_6 --> ctg2_7 --> ctg2_8 --> ctg2_9 --> ct...
2,2,MGYG000296008_2,-,ctg2_2 <-- ctg2_3 <-- ctg2_4 <-- ctg2_5
3,3,MGYG000296008_2,-,ctg2_14 <-- ctg2_15 <-- ctg2_16 <-- ctg2_17 <-...
4,4,MGYG000296008_22,-,ctg22_2 <-- ctg22_3 <-- ctg22_4 <-- ctg22_5 <-...
...,...,...,...,...
366,366,MGYG000296076_4,-,ctg4_74 <-- ctg4_75
367,367,MGYG000296076_4,-,ctg4_80 <-- ctg4_81
368,368,MGYG000296076_4,-,ctg4_85 <-- ctg4_86 <-- ctg4_87
369,369,MGYG000296076_4,-,ctg4_92 <-- ctg4_93


In [46]:
df_poem_final = (
    df_poem
    .groupby(["metagenome_id", "strand"], as_index=False)
    .agg({
        "coordinates": lambda x: " / ".join(x)
    })
)

In [47]:
df_poem_final

,metagenome_id,strand,coordinates
0,MGYG000296006_71,+,ctg71_3 --> ctg71_4 --> ctg71_5 --> ctg71_6 --...
1,MGYG000296008_2,+,ctg2_6 --> ctg2_7 --> ctg2_8 --> ctg2_9 --> ct...
2,MGYG000296008_2,-,ctg2_2 <-- ctg2_3 <-- ctg2_4 <-- ctg2_5 / ctg2...
3,MGYG000296008_22,-,ctg22_2 <-- ctg22_3 <-- ctg22_4 <-- ctg22_5 <-...
4,MGYG000296008_6,+,ctg6_37 --> ctg6_38 --> ctg6_39 --> ctg6_40 --...
...,...,...,...
159,MGYG000296076_1,-,ctg1_139 <-- ctg1_140 <-- ctg1_141 / ctg1_148 ...
160,MGYG000296076_10,+,3/548 --> ctg10_23 --> ctg10_24 --> ctg10_25
161,MGYG000296076_10,-,ctg10_29 <-- ctg10_30 <-- ctg10_31 <-- ctg10_3...
162,MGYG000296076_4,+,ctg4_63 --> ctg4_64 --> ctg4_65 --> ctg4_66 / ...


### Part 2 - Merging the Dataframes
Merging the df_poem_final (with operon information) with df_bgc (that contains the bgcs information).

In [49]:
df_bgc = pd.read_csv("/home/pedro/backup/important_tables/df_with_BGCs_information.csv")
df_bgc = df_bgc.rename(columns={"BGC": "metagenome_id"})
df_bgc

,metagenome_id,Reference_BGC,distance,jaccard,adjacency,dss,reference_BGC_class,reference_compound_name,has_resistance_protein,resistance_protein_class,resistance_protein_count,antismash_annotation,num_proteins_antismash_annotation
0,MGYG000296006_405,BGC0002398,0.47,1.00,1.00,0.38,terpene,(-)-ent-quiannulatene,False,NaN,NaN,NaN,NaN
1,MGYG000296006_71,BGC0002132,0.74,0.67,0.50,0.14,ribosomal,group 1 methanobactin,False,NaN,NaN,NaN,NaN
2,MGYG000296008_2,BGC0002283,0.71,0.50,0.40,0.23,terpene,sodorifen,False,NaN,NaN,NaN,NaN
3,MGYG000296008_22,BGC0000924,0.81,0.50,0.33,0.10,other,pyrrolnitrin,False,NaN,NaN,NaN,NaN
4,MGYG000296008_6,BGC0003157,0.82,0.50,0.00,0.10,terpene,Talaropentaene,False,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,MGYG000296075_3,BGC0001198,0.79,0.50,0.33,0.12,terpene,(+)-O-methylkolavelool,False,NaN,NaN,NaN,NaN
101,MGYG000296075_4,BGC0000536,0.71,0.67,0.33,0.18,ribosomal,nisin Q,True,glycopeptide,1.0,NaN,NaN
102,MGYG000296076_1,BGC0000656,0.91,0.33,0.00,0.04,terpene,zeaxanthin,False,NaN,NaN,NaN,NaN
103,MGYG000296076_10,BGC0000647,0.60,1.00,1.00,0.19,terpene,carotenoid,False,NaN,NaN,carotenoid,3.0


In [50]:
# Merging df_poem_final and df_bgc by metagenome_id column
df_final = df_bgc.merge(df_poem_final, on="metagenome_id", how="left")
df_final

,metagenome_id,Reference_BGC,distance,jaccard,adjacency,dss,reference_BGC_class,reference_compound_name,has_resistance_protein,resistance_protein_class,resistance_protein_count,antismash_annotation,num_proteins_antismash_annotation,strand,coordinates
0,MGYG000296006_405,BGC0002398,0.47,1.00,1.00,0.38,terpene,(-)-ent-quiannulatene,False,NaN,NaN,NaN,NaN,NaN,NaN
1,MGYG000296006_71,BGC0002132,0.74,0.67,0.50,0.14,ribosomal,group 1 methanobactin,False,NaN,NaN,NaN,NaN,+,ctg71_3 --> ctg71_4 --> ctg71_5 --> ctg71_6 --...
2,MGYG000296008_2,BGC0002283,0.71,0.50,0.40,0.23,terpene,sodorifen,False,NaN,NaN,NaN,NaN,+,ctg2_6 --> ctg2_7 --> ctg2_8 --> ctg2_9 --> ct...
3,MGYG000296008_2,BGC0002283,0.71,0.50,0.40,0.23,terpene,sodorifen,False,NaN,NaN,NaN,NaN,-,ctg2_2 <-- ctg2_3 <-- ctg2_4 <-- ctg2_5 / ctg2...
4,MGYG000296008_22,BGC0000924,0.81,0.50,0.33,0.10,other,pyrrolnitrin,False,NaN,NaN,NaN,NaN,-,ctg22_2 <-- ctg22_3 <-- ctg22_4 <-- ctg22_5 <-...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,MGYG000296076_1,BGC0000656,0.91,0.33,0.00,0.04,terpene,zeaxanthin,False,NaN,NaN,NaN,NaN,-,ctg1_139 <-- ctg1_140 <-- ctg1_141 / ctg1_148 ...
166,MGYG000296076_10,BGC0000647,0.60,1.00,1.00,0.19,terpene,carotenoid,False,NaN,NaN,carotenoid,3.0,+,3/548 --> ctg10_23 --> ctg10_24 --> ctg10_25
167,MGYG000296076_10,BGC0000647,0.60,1.00,1.00,0.19,terpene,carotenoid,False,NaN,NaN,carotenoid,3.0,-,ctg10_29 <-- ctg10_30 <-- ctg10_31 <-- ctg10_3...
168,MGYG000296076_4,BGC0002865,0.56,1.00,1.00,0.26,PKS,Aloesone,False,NaN,NaN,NaN,NaN,+,ctg4_63 --> ctg4_64 --> ctg4_65 --> ctg4_66 / ...


In [51]:
df_final.to_csv("updated_BGCs_information.csv")